# 🗳️ Analyse des Résultats Présidentiels 2022 - Lyon (Couche Gold BI)

Ce notebook permet de visualiser et de valider les données de la couche **Gold BI**. Nous utilisons le schéma en étoile composé des dimensions `dim_candidats`, `dim_geographie` et des tables de faits `fact_votes` et `fact_participation`.

In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from src.common.spark_session_manager import get_spark_session
from src.config import GOLD_PATH

spark = get_spark_session("Notebook_Gold_BI_Viz")
spark.conf.set("spark.sql.execution.arrow.pyspark.enabled", "true")
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = [12, 6]

path_gold = os.path.join(GOLD_PATH, "presidentielle", "bi")

## 1. Chargement des données Gold

In [ ]:
dim_candidats = spark.read.parquet(os.path.join(path_gold, "dim_candidats"))
dim_geographie = spark.read.parquet(os.path.join(path_gold, "dim_geographie"))
fact_votes = spark.read.parquet(os.path.join(path_gold, "fact_votes"))
fact_participation = spark.read.parquet(os.path.join(path_gold, "fact_participation"))

## 2. Analyse Globale par Candidat (Tour 1 vs Tour 2)

In [ ]:
df_viz = fact_votes.join(dim_candidats, "id_candidat") \
    .groupBy("nom", "prenom", "tour") \
    .agg(F.sum("voix").alias("total_voix"))

pdf_viz = df_viz.toPandas()
pdf_viz['candidat'] = pdf_viz['prenom'] + " " + pdf_viz['nom']

fig, ax = plt.subplots(1, 2, figsize=(18, 7), sharey=True)
for t in [1, 2]:
    data_tour = pdf_viz[pdf_viz['tour'] == t].sort_values('total_voix', ascending=False)
    sns.barplot(data=data_tour, x='total_voix', y='candidat', ax=ax[t-1], palette='viridis')
    ax[t-1].set_title(f"Résultats du Tour {t}")
    ax[t-1].set_xlabel("Nombre de Voix")

plt.tight_layout()
plt.show()

## 3. Analyse Géo-Politique : Composition par Blocs (T1 vs T2)

Cette vue permet d'observer le passage du pluralisme du 1er tour au duel final du 2nd tour.

In [ ]:
# Calcul des voix par arrondissement, par tour et par bloc
df_blocs_arr = fact_votes \
    .join(dim_candidats, "id_candidat") \
    .join(dim_geographie, "id_bureau") \
    .groupBy("arrondissement", "tour", "bloc_analytique") \
    .agg(F.sum("voix").alias("total_voix"))

pdf_blocs = df_blocs_arr.orderBy("arrondissement", "tour").toPandas()

# Visualisation comparative en volumes aboslu
fig, ax = plt.subplots(1, 2, figsize=(20, 8), sharey=True)

for t in [1, 2]:
    data_t = pdf_blocs[pdf_blocs['tour'] == t]
    sns.histplot(
        data=data_t, x='arrondissement', hue='bloc_analytique',
        weights='total_voix', multiple='stack', shrink=.8, ax=ax[t-1]
    )
    ax[t-1].set_title(f"Composition Politique (Volumes) - Tour {t}")
    ax[t-1].tick_params(axis='x', rotation=45)
    ax[t-1].set_ylabel("Cumul des voix")
    ax[t-1].set_xlabel("")

plt.tight_layout()
plt.show()

### 3b. Analyse de la Structure Politique (Normalisée à 100%)

Cette vue permet de comparer la "sensibilité" politique des arrondissements en faisant abstraction de leur taille démographique. Chaque barre représente 100% des voix exprimées.

In [ ]:
# Visualisation comparative (relative)
fig, ax = plt.subplots(1, 2, figsize=(20, 8), sharey=True)

for t in [1, 2]:
    data_t = pdf_blocs[pdf_blocs['tour'] == t]
    sns.histplot(
        data=data_t, x='arrondissement', hue='bloc_analytique',
        weights='total_voix', multiple='fill', shrink=.8, ax=ax[t-1]
    )
    ax[t-1].set_title(f"Structure Politique (Proportions 100%) - Tour {t}")
    ax[t-1].tick_params(axis='x', rotation=45)
    ax[t-1].set_ylabel("Proportion des voix")
    ax[t-1].set_xlabel("")

plt.tight_layout()
plt.show()

## 4. Évolution de la Participation

Indicateur de la mobilisation électorale locale.

In [ ]:
df_participation = fact_participation.join(dim_geographie, "id_bureau") \
    .groupBy("arrondissement", "tour") \
    .agg(
        F.sum("inscrits").alias("total_inscrits"),
        F.sum("votants").alias("total_votants")
    ) \
    .withColumn("taux_participation", (F.col("total_votants") / F.col("total_inscrits")) * 100)

pdf_part = df_participation.orderBy("arrondissement", "tour").toPandas()

# Visualisation par Barres Groupées
plt.figure(figsize=(14, 7))
sns.barplot(data=pdf_part, x='arrondissement', y='taux_participation', hue='tour', palette='muted')
plt.title("Taux de Participation par Arrondissement (Tour 1 vs Tour 2)")
plt.ylabel("% Participation")
plt.ylim(60, 95)
plt.legend(title='Tour', loc='upper right')
plt.xticks(rotation=45)
plt.show()

## 5. Synthèse Communale (Lyon Global)

Répartition des forces politiques à l'échelle de la ville entière sous forme de camembert pour visualiser les parts de voix.

In [ ]:
# Agrégation par tour et par bloc
df_lyon_global = fact_votes \
    .join(dim_candidats, "id_candidat") \
    .groupBy("tour", "bloc_analytique") \
    .agg(F.sum("voix").alias("total_voix_lyon"))

pdf_lyon = df_lyon_global.orderBy("tour", F.col("total_voix_lyon").desc()).toPandas()

# Camemberts des nuances politiques
fig, ax = plt.subplots(1, 2, figsize=(20, 10))
colors = sns.color_palette('pastel')

for t in [1, 2]:
    data_t = pdf_lyon[pdf_lyon['tour'] == t]
    ax[t-1].pie(
        data_t['total_voix_lyon'],
        labels=data_t['bloc_analytique'],
        autopct='%1.1f%%',
        startangle=140,
        colors=colors,
        textprops={'fontsize': 12}
    )
    ax[t-1].set_title(f"Répartition des Voix à Lyon - Tour {t}", fontsize=15, fontweight='bold')

plt.tight_layout()
plt.show()

## 6. Audit de Qualité Final

In [ ]:
audit_df = fact_votes.join(dim_candidats, "id_candidat") \
    .select("id_candidat", "nom", "prenom") \
    .distinct() \
    .groupBy("id_candidat") \
    .count() \
    .filter("count > 1")

if audit_df.count() == 0:
    print("✅ Audit OK : Intégrité référentielle parfaite.")
else:
    print("❌ ERREUR : Doublons d'id détectés !")
    audit_df.show()